In [1]:
import pandas as pd
import ast
from pathlib import Path
from config import RESULTS_DIR

2026-01-26 17:19:20.285 | INFO     | config:<module>:6 - PROJ_ROOT path is: /home/navarri/AtriaProject/deepcsdf-atria


In [4]:
# some helpers
def parse_value(x):
    if isinstance(x, str) and x.startswith("["): # sometimes I saved values in list, likely from LDDMM output
        return ast.literal_eval(x)[0]
    return float(x)

In [9]:
fname = RESULTS_DIR / "metrics" / "version_0-LDDMM-trainshapes.parquet"
df = pd.read_parquet(fname)
df["value"] = df["value"].apply(parse_value)
df

,version,patient,organ,metric,value
0,0,AF059,epicardium,LDDMM,0.003897
1,0,AF059,la_endo,LDDMM,0.001709
2,0,AF059,ra_endo,LDDMM,0.006599
3,0,LEU_NORM_F017,epicardium,LDDMM,0.006324
4,0,LEU_NORM_F017,la_endo,LDDMM,0.003220
5,0,LEU_NORM_F017,ra_endo,LDDMM,0.002290
6,0,AF035,epicardium,LDDMM,0.040154
7,0,AF035,la_endo,LDDMM,0.004505
8,0,AF035,ra_endo,LDDMM,0.051665
9,0,AF019,epicardium,LDDMM,0.013397


Mean error per version, per organ, across patients

In [ ]:
df.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

mean    median       std       q95
version metric organ                                             
0       LDDMM  epicardium  0.152249  0.018055  0.422739  0.764673
               la_endo     0.049353  0.013098  0.100502  0.210728
               ra_endo     0.011560  0.006218  0.015341  0.036802

Mean error per version, across organs, across patients : one single number per version. Organs are weighted the same here.

In [11]:
df.groupby(["version", "metric"])["value"].agg(
    mean="mean",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

,,mean,std,q95
version,metric,,,
0,LDDMM,0.071054,0.24965,0.210728


# Load several versions / metrics

In [12]:
# retrieve all the wanted files
dfs = []

for file_path in Path("results/metrics").glob("version_*.parquet"):
    # extract version from filename, e.g. version_89.csv → 89
    version = file_path.stem.split("-")[0].split("_")[-1]
    df = pd.read_parquet(file_path)
    df["version"] = int(version)
    df["value"] = df["value"].apply(parse_value)
    dfs.append(df)

df_across_vers = pd.concat(dfs, ignore_index=True)
# df_across_vers["version"].unique() # inspect which versions are there

In [13]:
df_across_vers.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

mean    median       std       q95
version metric  organ                                             
0       LDDMM   epicardium  0.152249  0.018055  0.422739  0.764673
                la_endo     0.049353  0.013098  0.100502  0.210728
                ra_endo     0.011560  0.006218  0.015341  0.036802
        chamfer epicardium  0.022101  0.016298  0.010832  0.038382
                la_endo     0.032072  0.023782  0.017901  0.058494
                ra_endo     0.025818  0.018489  0.014907  0.048269

# Ranking versions